In [1]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
#this is done in order to import tensorflow then keras and later we will use keras's layer ssystem in our model.

In [2]:

i_s = (224,224)
b_s = 32
#these are image size and batch(number of images in one epoch) size respectively.
dir = r"E:\Downloads\Telegram Desktop\plant_disease\all\Crop Diseases\tomato"
# now creating treaning set.
train = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "training",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
    
val = keras.utils.image_dataset_from_directory(
    dir,
    validation_split = 0.20,
    subset = "validation",
    seed = 123,
    image_size = i_s,
    batch_size = b_s,
    color_mode = "rgb"
)
#this is the validation set
#now time for dividing validation set into val and test sets
valbatches = tf.data.experimental.cardinality(val)
test = val.take(valbatches//2)
val= val.skip(valbatches//2)
class_names = train.class_names
print("Classes:", class_names)
print("Train batches:", tf.data.experimental.cardinality(train).numpy())
print("Val batches:", tf.data.experimental.cardinality(val).numpy())
print("Test batches:", tf.data.experimental.cardinality(test).numpy())
# it is better to shuffle trianing set before augmentation
train = train.shuffle(1000).prefetch(buffer_size = tf.data.AUTOTUNE)
#previous line means it shuffle in batch of 1000 and cpu automatically loads the next batch while gpu is training model by using prefetch function
val   = val.prefetch(buffer_size=tf.data.AUTOTUNE)
test  = test.prefetch(buffer_size=tf.data.AUTOTUNE)
#now its time for augmentation
aug = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import optimizers , models, layers
num = len(class_names)

base = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base.trainable = False
from keras.callbacks import EarlyStopping , ReduceLROnPlateau , ModelCheckpoint
inputs = layers.Input((224, 224, 3))
x = aug(inputs)

x = base(x, training = False)
#this is the phase of augmentation and normalization like max norm

x = layers.GlobalAveragePooling2D()(x) #this is pooling
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)
outputs = layers.Dense(num, activation = "softmax")(x)
model = models.Model(inputs, outputs)
model.compile(
    optimizer=optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)
callbacks = [
    EarlyStopping(patience=3, restore_best_weights=True),
    ReduceLROnPlateau(patience=2, factor=0.3),
    ModelCheckpoint("models/head_stage.h5", save_best_only=True)
]
history_head = model.fit(
    train,
    validation_data=val,
    epochs=10,
    callbacks=callbacks
)


Found 8283 files belonging to 8 classes.
Using 6627 files for training.
Found 8283 files belonging to 8 classes.
Using 1656 files for validation.
Classes: ['Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus']
Train batches: 208
Val batches: 26
Test batches: 26
Epoch 1/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5244 - loss: 1.4655

208/208 ━━━━━━━━━━━━━━━━━━━━ 352s 1s/step - accuracy: 0.6754 - loss: 0.9732 - val_accuracy: 0.8337 - val_loss: 0.6094 - learning_rate: 0.0010
Epoch 2/10
144/208 ━━━━━━━━━━━━━━━━━━━━ 1:24 1s/step - accuracy: 0.8079 - loss: 0.5611

208/208 ━━━━━━━━━━━━━━━━━━━━ 360s 2s/step - accuracy: 0.8269 - loss: 0.5040 - val_accuracy: 0.8738 - val_loss: 0.3723 - learning_rate: 0.0010
Epoch 3/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8603 - loss: 0.4129

208/208 ━━━━━━━━━━━━━━━━━━━━ 327s 1s/step - accuracy: 0.8606 - loss: 0.4189 - val_accuracy: 0.8956 - val_loss: 0.3001 - learning_rate: 0.0010
Epoch 4/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 969ms/step - accuracy: 0.8771 - loss: 0.3530

208/208 ━━━━━━━━━━━━━━━━━━━━ 278s 1s/step - accuracy: 0.8757 - loss: 0.3633 - val_accuracy: 0.8908 - val_loss: 0.2854 - learning_rate: 0.0010
Epoch 5/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 274s 1s/step - accuracy: 0.8892 - loss: 0.3243 - val_accuracy: 0.8981 - val_loss: 0.3015 - learning_rate: 0.0010
Epoch 6/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8925 - loss: 0.3063

208/208 ━━━━━━━━━━━━━━━━━━━━ 283s 1s/step - accuracy: 0.8933 - loss: 0.3127 - val_accuracy: 0.9029 - val_loss: 0.2686 - learning_rate: 0.0010
Epoch 7/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.8911 - loss: 0.3037

208/208 ━━━━━━━━━━━━━━━━━━━━ 297s 1s/step - accuracy: 0.8864 - loss: 0.3243 - val_accuracy: 0.9248 - val_loss: 0.2232 - learning_rate: 0.0010
Epoch 8/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 294s 1s/step - accuracy: 0.8926 - loss: 0.3171 - val_accuracy: 0.9211 - val_loss: 0.2468 - learning_rate: 0.0010
Epoch 9/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 306s 1s/step - accuracy: 0.8957 - loss: 0.2942 - val_accuracy: 0.9296 - val_loss: 0.2315 - learning_rate: 0.0010
Epoch 10/10
208/208 ━━━━━━━━━━━━━━━━━━━━ 0s 959ms/step - accuracy: 0.9013 - loss: 0.2865

208/208 ━━━━━━━━━━━━━━━━━━━━ 273s 1s/step - accuracy: 0.9033 - loss: 0.2805 - val_accuracy: 0.9296 - val_loss: 0.2207 - learning_rate: 3.0000e-04


In [3]:
model.save("tomato.keras")

In [4]:
test_loss, test_accuracy = model.evaluate(test)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy * 100:.2f}%")

26/26 ━━━━━━━━━━━━━━━━━━━━ 23s 864ms/step - accuracy: 0.9207 - loss: 0.2258
Test Loss: 0.2258
Test Accuracy: 92.07%
